#
[Create SQL Functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-and-use-a-sql-scalar-function)
#
[Create Python Functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-python-functions)
# 
[Create Custom Tools](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#create-tool)

In [1]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_PERSONAL_ACCESS_TOKEN")

In [ ]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

# Serverless mode for production
client = DatabricksFunctionClient(execution_mode="serverless")

# Local mode for development
client = DatabricksFunctionClient(execution_mode="local")

In [3]:
CATALOG = "workspace"
SCHEMA = "default"

In [4]:
def add_numbers(number_1: float, number_2: float) -> float:
  """
  A function that accepts two floating point numbers adds them,
  and returns the resulting sum as a float.

  Args:
    number_1 (float): The first of the two numbers to add.
    number_2 (float): The second of the two numbers to add.

  Returns:
    float: The sum of the two input numbers.
  """
  return number_1 + number_2

function_info = client.create_python_function(
  func=add_numbers,
  catalog=CATALOG,
  schema=SCHEMA,
  replace=True
)

In [5]:
result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.add_numbers",
  parameters={"number_1": 36939.0, "number_2": 8922.4}
)

result.value # OUTPUT: '45861.4'

'45861.4'

In [8]:
# workspace.default.custom_divide

result = client.execute_function(
  function_name=f"{CATALOG}.{SCHEMA}.custom_divide",
  parameters={"n1": 36939, "n2": 3}
)

result.value

In [13]:
from agents import Agent, Runner
from databricks.sdk import WorkspaceClient
from databricks_openai.agents import McpServer
from langchain.agents import create_agent
workspace_client = WorkspaceClient()

async with McpServer.from_uc_function(
    catalog=CATALOG,
    schema=SCHEMA,
    workspace_client=workspace_client,
    name="uc-functions",
) as uc_server:
    # agent = create_agent(
    #     ChatDatabricks(endpoint="databricks-gemma-3-12b"),
    #     instructions="You are a helpful assistant. Use the available tools to answer questions.",
    #     mcp_servers=[uc_server],
    # )

    agent = Agent(
        name="Dataabricks Agent",
        instructions="You are a helpful assistant. Use the available tools to answer questions.",
        model="gemma4:latest", #databricks-claude-sonnet-4-5 #databricks-gemma-3-12b #gemma4:latest
        mcp_servers=[uc_server],
    )
    result = await Runner.run(agent, "add two numbers 36939 and 11 ")
    print(result.final_output)

Error getting response: Error code: 400 - {'error': {'message': "The requested model 'gemma4:latest' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}. (request_id: req_62f3966d59b34c1a84a59da3cb4f9828)


BadRequestError: Error code: 400 - {'error': {'message': "The requested model 'gemma4:latest' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}

#### Using UCFunctionToolkit

In [14]:
from databricks_langchain import UCFunctionToolkit

# Create a toolkit with the Unity Catalog function
func_name = f"{CATALOG}.{SCHEMA}.add_numbers"
toolkit = UCFunctionToolkit(function_names=[func_name])

tools = toolkit.tools

In [18]:
from langchain.agents import create_agent 
from langchain_classic.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain import (
  ChatDatabricks,
  UCFunctionToolkit,
)
import mlflow

# Initialize the LLM (optional: replace with your LLM of choice)
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.1)

# Define the prompt
prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful assistant. Make sure to use tools for additional functionality.",
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
  ]
)

# Enable automatic tracing
mlflow.langchain.autolog()

# Define the agent, specifying the tools from the toolkit above
agent = create_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "What is 36939.0 + 8922.4?"})

TypeError: create_agent() takes from 1 to 2 positional arguments but 3 were given